Hi Diego! Here I'm simulating some data that combines the types of variables you might have in your dataset.

Imagine that each row in the dataset represents one patent, and we are asking a LLM to provide the following information:

* `is_relevant`: this can be "yes" is the patent is about heat pumps, "no" otherwise.

* `non_trad_categories`: this is a list of any non-traditional technologies mentioned in the patent. It can be any combination of

```
[
    "Elastocaloric", "Electrocaloric", "Magnetocaloric", "Ionocaloric",
    "Barocaloric", "Thermoelectric", "Electrochemical", "Thermoacoustic", "Other"
]
```
... and it can be an empty list `[]` if none of these apply.

* `score`: this is a score from 0-5 (imagine this is a score for cost, efficiency, or circularity).

For each of these variables we have a machine-generated output and we also have a human rating, so there are two columns per variable.

In [ ]:
import random

import pandas as pd

# Seed for reproducibility
random.seed(42)

categories = [
    "Elastocaloric",
    "Electrocaloric",
    "Magnetocaloric",
    "Ionocaloric",
    "Barocaloric",
    "Thermoelectric",
    "Electrochemical",
    "Thermoacoustic",
    "Other",
]


def random_categories() -> list[str]:
    return random.sample(categories, random.randint(0, 3))


def random_score() -> int:
    return random.randint(0, 5)


def random_binary() -> str:
    return random.choice(["yes", "no"])


# Create dummy data
n = 1000
df = pd.DataFrame(
    {
        "is_relevant_human": [random_binary() for _ in range(n)],
        "is_relevant_machine": [random_binary() for _ in range(n)],
        "non_trad_categories_human": [random_categories() for _ in range(n)],
        "non_trad_categories_machine": [random_categories() for _ in range(n)],
        "score_human": [random_score() for _ in range(n)],
        "score_machine": [random_score() for _ in range(n)],
    }
)

df.head()

# Evaluation of a binary variable (relevance)

First we'll calculate metrics for the binary variable `is_relevant`.

Because we generated the distribution at random, you can see that in our dummy data the machine is more or less 50/50 at predicting the relevance of a patent. We'd hope that in real life the LLM would be better!

In [ ]:
df.groupby(["is_relevant_human", "is_relevant_machine"]).size()

One nice way to visualise this (and this can be helpful especially if there are more categories than 2) is with a confusion matrix. You hope to get most values along the diagonal (i.e. true and predicted values align) and not so many off the diagonal. But in our case, with our random data, there are similar counts in every cell!

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Convert to binary 0/1
y_true = df["is_relevant_human"].map({"yes": 1, "no": 0})
y_pred = df["is_relevant_machine"].map({"yes": 1, "no": 0})

# Generate confusion matrix
cm = confusion_matrix(y_true, y_pred)
labels = ["No", "Yes"]

# Plot
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix: is_relevant")
plt.show()

Now to quantify this performance, the typical metrics used are **accuracy** (how many did the machine get right overall), **recall** (true positive rate), **precision** (false positive rate), and **F1 score** which is the harmonic mean of precision and recall.

You can read more about these here:
* https://developers.google.com/machine-learning/crash-course/classification/accuracy-precision-recall
* https://www.coursera.org/articles/precision-vs-recall-machine-learning

Scores close to 1 are better! Again, because we generated the data randomly, the machine gets scores close to 50% on all metrics.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

print("Binary Classification Metrics:")
print("Accuracy:", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred))
print("F1 Score:", f1_score(y_true, y_pred))

# Evaluating a multi-class variable (non-traditional categories)

This variable is a bit more complicated because each patent could have any combination of non-traditional technologies, or none of them.

We can still use confusion matrices to help us understand how the machine has performed, but it needs to be one confusion matrix per category as each patent could relate to multiple categories.

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

# NB we use `MultiLabelBinarizer` to transform the columns that contain lists of strings into matrices.
# If your data just has one column per subdomain, this step can be skipped.
mlb = MultiLabelBinarizer(classes=categories)

y_true_multilabel = mlb.fit_transform(df["non_trad_categories_human"])
y_pred_multilabel = mlb.transform(df["non_trad_categories_machine"])

# Plot confusion matrix for each category
for i, class_name in enumerate(mlb.classes_):
    cm = confusion_matrix(y_true_multilabel[:, i], y_pred_multilabel[:, i])

    plt.figure(figsize=(4, 3))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["No", "Yes"],
        yticklabels=["No", "Yes"],
    )
    plt.title(f"Confusion Matrix for {class_name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()

If you looked at confusion matrices like these and found that the machine was over-assigning one category or rarely/never using another category, you could consider expanding the prompt a little bit to explain what the categories mean and how they should be used.

In reality, you will have a lot of categories and we probably won't have time to label and look in detail at all of them! However, it is still possible to get an overall metric for the variable without looking in depth at each category within that variable. The code below averages out the performance across different categories within the variable `non_trad_categories`:

In [ ]:
print("\nMultilabel Classification Metrics (macro avg):")
print(
    "Precision:", precision_score(y_true_multilabel, y_pred_multilabel, average="macro")
)
print("Recall:", recall_score(y_true_multilabel, y_pred_multilabel, average="macro"))
print("F1 Score:", f1_score(y_true_multilabel, y_pred_multilabel, average="macro"))

# Evaluating a numeric score

Below I have given some code that gives you one way to check the difference in distributions between human-labelled and machine-labelled scores:

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

# Plot histograms
plt.hist(
    df["score_human"],
    bins=6,
    alpha=0.6,
    label="Human Score",
    color="skyblue",
    edgecolor="black",
)
plt.hist(
    df["score_machine"],
    bins=6,
    alpha=0.6,
    label="Machine Score",
    color="salmon",
    edgecolor="black",
)

# Formatting
plt.xlabel("Score")
plt.ylabel("Frequency")
plt.title("Comparison of Human vs Machine Scores")
plt.legend()
plt.xticks(range(0, 6))  # scores from 0 to 5
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

... but if in reality there is not time to manually score lots of examples, it might still be worth checking eve just the distribution of machine generated scores? You can do that very easily like this:

In [ ]:
df["score_machine"].hist()

You would be concerned if, for example, the LLM never used one of the scores or if it was always using the same score.

If you do have enough human-labelled scores, we can generate metrics!

We can choose from:

* Mean Absolute Error (MAE): the average absolute difference between human-labelled values and machine-predicted values.
* Mean Squared Error (MSE): the average squared difference between human-labelled values and machine-predicted values.
* R-squared: the proportion of variance in the human-labelled values that can be explained by the machine-predicted values. If it is negative, it means you would have done better just using the mean score every time!

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_true_score = df["score_human"]
y_pred_score = df["score_machine"]

print("\nNumeric Prediction Metrics:")
print("MAE:", mean_absolute_error(y_true_score, y_pred_score))
print("MSE:", mean_squared_error(y_true_score, y_pred_score))
print("R^2:", r2_score(y_true_score, y_pred_score))